# Tool System — Kaggle Bootstrap

Notebook mới cho repository Tool System V2. Không import notebook cũ của Money AI, không đọc secrets từ dự án cũ và không khởi chạy Chrome.

Notebook này chỉ clone mã nguồn công khai, chạy unit tests và kiểm tra xem Chrome CDP `127.0.0.1:9222` đã tồn tại hay chưa. Nếu CDP không có, notebook báo rõ và dừng ở trạng thái chưa xác minh.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/duykhanhlongchutayninh-debug/tool-system-v2.git"
WORKDIR = Path("/kaggle/working/tool-system-v2")

if WORKDIR.exists():
    print(f"Repository directory already exists: {WORKDIR}")
    print("Not deleting or overwriting it. Review its state before proceeding.")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORKDIR)], check=True)

print("Python:", sys.version)
print("Working directory:", WORKDIR)

In [ ]:
%cd /kaggle/working/tool-system-v2
!python -m pip install -e .
!python -m unittest discover -s tests -v

In [ ]:
import json
from urllib.request import urlopen
from urllib.error import URLError

CDP_VERSION_URL = "http://127.0.0.1:9222/json/version"
try:
    with urlopen(CDP_VERSION_URL, timeout=3) as response:
        if response.status != 200:
            raise RuntimeError(f"CDP returned HTTP {response.status}")
        data = json.loads(response.read().decode("utf-8"))
    if not data.get("Browser") or not str(data.get("webSocketDebuggerUrl", "")).startswith("ws"):
        raise RuntimeError("CDP response is missing required browser metadata")
    print(json.dumps({"status": "PASS", "browser": data["Browser"]}, ensure_ascii=False))
except (URLError, TimeoutError, OSError, RuntimeError, json.JSONDecodeError) as exc:
    print(json.dumps({"status": "NOT_VERIFIED", "reason": str(exc), "chrome_started": False}, ensure_ascii=False))
    print("This notebook does not launch Chrome. Confirm the intended existing Chrome/CDP setup before continuing.")

## Giới hạn an toàn

- Không có lệnh đăng Facebook hoặc kích hoạt lịch đăng thật.
- Không tạo Chrome mới.
- Không đưa token, cookie hay thông tin đăng nhập vào notebook.
- Kaggle session phải còn chạy thì các tiến trình trong session mới hoạt động; notebook này không tạo lịch chạy 24/7.
- Kết quả clone/test/CDP chỉ được công nhận sau khi chạy cell và xem output thực tế.